# Part 4: Combine all knowledge sources

Part 4 at Caldova begins when an executive asks for a supply-chain briefing on `CALD-201`, current Summit Dose actions, purchasing controls, and medicinal-product dependencies. You need one answer that combines all available sources while preserving the boundary between policy, contractual evidence, structured Fabric facts, and workplace discussions.

**🎯 Mission**
- Build one **five-source knowledge base** with the policy file, both Search indexes, Fabric IQ, and Work IQ
- Prepare an executive supply-chain briefing with each claim attributed to the correct evidence type
- Avoid inferring a Summit Dose-to-product relationship unless Fabric IQ explicitly provides it

## Step 1: Set up environment variables

Load the configuration for your Foundry IQ (Azure AI Search), Azure OpenAI, and Fabric resources. The `.env` file in the project folder has the values needed for this part.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-08-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
WORK_IQ_ENTRA_APP_ID = os.environ["WORK_IQ_ENTRA_APP_ID"]
WORK_IQ_FEDERATED_CREDENTIAL_ID = os.environ["WORK_IQ_FEDERATED_CREDENTIAL_ID"]
WORK_IQ_ENTRA_TENANT_ID = os.getenv("WORK_IQ_ENTRA_TENANT_ID", AZURE_TENANT_ID)
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

SUPPLIER_EVIDENCE_INDEX = "supplier-evidence"
SOURCING_DOCUMENTS_INDEX = "sourcing-documents"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

## Step 2: Login and acquire delegated token

In order for knowledge base retrieval to use your identity with Work IQ and Fabric IQ knowledge sources, you need an OAuth token for a signed-in user who has access to both Microsoft 365 data and the Fabric workspace.

Run the cell below to sign in using the credentials in the instructions sidebar. If `FABRIC_TENANT_ID` is configured, authentication uses that tenant. Otherwise, it uses `AZURE_TENANT_ID`. A successful sign-in acquires a delegated token for the Foundry IQ (Azure AI Search) scope and saves it for use as `query_source_authorization` during retrieval. It also acquires a Work IQ user assertion for the `access_as_user` scope on your Work IQ Entra app, saved for use as `query_work_iq_source_authorization`.

In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(
    tenant_id=os.getenv("FABRIC_TENANT_ID", AZURE_TENANT_ID)
)
user_token = user_credential.get_token("https://search.azure.com/.default").token

workiq_credential = InteractiveBrowserCredential(
    client_id=WORK_IQ_ENTRA_APP_ID,
    tenant_id=WORK_IQ_ENTRA_TENANT_ID,
)
user_assertion = workiq_credential.get_token(
    f"api://{WORK_IQ_ENTRA_APP_ID}/access_as_user"
).token

print("Acquired delegated token and Work IQ user assertion for logged-in user")

## Step 3: Create all knowledge sources

Reuse the direct policy file source from Part 1, recreate both indexed source definitions, and add Work IQ and Fabric IQ:

* `file-docs-knowledge-source`: Caldova's uploaded purchasing policy
* `supplier-evidence-knowledge-source`: Supplier invoices, shipment evidence, production yield records, and quality records
* `sourcing-documents-knowledge-source`: Approved RFP responses, agreements, amendments, and purchase orders
* `workiq-knowledge-source`: Current workplace actions and decisions
* `fabric-ontology-knowledge-source`: Structured manufacturer, active-substance, and medicinal-product relationships

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    EntraAppAuthentication,
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
    WorkIQKnowledgeSourceParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"
SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME = "supplier-evidence-knowledge-source"
SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME = "sourcing-documents-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    FILE_KNOWLEDGE_SOURCE_NAME,
    SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
    SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential, api_version=AZURE_SEARCH_API_VERSION)

for knowledge_source_name, index_name, description in [
    (
        SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
        SUPPLIER_EVIDENCE_INDEX,
        "Caldova supplier invoices, shipment evidence, production yield records, and quality records",
    ),
    (
        SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
        SOURCING_DOCUMENTS_INDEX,
        "Caldova approved RFP responses, agreements, amendments, and purchase orders",
    ),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="title"),
                SearchIndexFieldReference(name="page_number_from"),
                SearchIndexFieldReference(name="page_number_to"),
            ],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Caldova workplace context for current sourcing actions and decisions",
    work_iq_parameters=WorkIQKnowledgeSourceParameters(
        entra_app_authentication=EntraAppAuthentication(
            application_id=WORK_IQ_ENTRA_APP_ID,
            federated_credential_id=WORK_IQ_FEDERATED_CREDENTIAL_ID,
            tenant_id=WORK_IQ_ENTRA_TENANT_ID,
        )
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Caldova Fabric ontology for manufacturer, active-substance, and medicinal-product relationships",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)
print("Knowledge sources created")

## Step 4: Create the combined knowledge base

The combined knowledge base uses `outputMode=answerSynthesis` and `retrievalReasoningEffort=low` to plan retrieval across all five sources.

Its instructions assign a clear role to each source: the file source provides purchasing policy, the Search indexes provide supplier and contractual evidence, Fabric IQ provides structured relationships, and Work IQ provides current workplace discussion. If Fabric IQ does not connect Summit Dose directly to a medicinal product, the answer must not infer that relationship.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "caldova-supply-chain-briefing-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova supply-chain briefing knowledge base combining policy, procurement evidence, Fabric relationships, and workplace context",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use the file source for purchasing policy, Search indexes for supplier and contractual evidence, Fabric Ontology for structured manufacturer, substance, and product relationships, and Work IQ for current workplace discussions, especially Summit Dose equipment mapping, Piedmont subcontractor approval, and PPQ-01 delivery planning. Clearly label each evidence type. Do not infer a Summit Dose-to-product relationship unless Fabric Ontology explicitly provides one.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

## Step 5: Query the knowledge base

Prepare one executive supply-chain briefing across all five sources:

* Purchasing controls and escalation responsibilities come from the direct policy file.
* Approved `CALD-201` commitments come from `sourcing-documents`.
* Current Summit Dose actions come from Work IQ.
* Manufacturer and medicinal-product dependencies come from Fabric IQ.
* Supporting operational evidence remains available from `supplier-evidence`.

The answer must distinguish policy, contractual evidence, structured Fabric facts, and workplace discussions. It must not infer a Summit Dose-to-product link when Fabric IQ does not provide one.

⏱️ The sample question takes about 1 to 2 minutes because it includes Work IQ and Fabric IQ.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential, api_version=AZURE_SEARCH_API_VERSION
)

question = (
    "Prepare a Caldova supply-chain briefing. Summarize the approved CALD-201 sourcing commitments and assess "
    "the current Summit Dose equipment-mapping action, Piedmont subcontractor approval, and PPQ-01 April delivery "
    "decision from my work context. Use Fabric IQ to identify medicinal-product dependencies for approved "
    "manufacturers that may create additional supply risk. Identify owners, deadlines, and unresolved decisions. "
    "Clearly distinguish policy, contractual evidence, structured Fabric facts, and workplace discussions."
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        FileKnowledgeSourceParams(
            knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
            reranker_threshold=0.0,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=300,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
    query_work_iq_source_authorization=user_assertion,
)

if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

For this activity log, you should see calls made to "fabricOntology", "workIQ", and "searchIndex", with queries for each step.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

The response should include a mix of `file`, `searchIndex`, `fabricOntology`, and `workIQ` references. Use them to verify that every briefing claim is attributed to policy, contractual or supplier evidence, a structured Fabric relationship, or workplace discussion.

If Fabric IQ returns medicinal-product dependencies for approved manufacturers but no Summit Dose relationship, report those dependencies independently and preserve that evidence boundary.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

Look at the references printed above. Can you identify:

1. Which file reference supports Caldova purchasing policy?
2. Which indexed references support approved `CALD-201` commitments and any supplier evidence?
3. Which Work IQ references support the equipment-mapping action, Piedmont approval status, and PPQ-01 delivery decision, including their owners and deadlines?
4. Which Fabric IQ results support manufacturer-to-substance and substance-to-product relationships?
5. Does the answer keep workplace updates separate from approved commitments and avoid claiming a Summit Dose-to-product relationship when Fabric IQ does not provide one?

## ✅ Mission complete

**What you built across the lab:**

* ✓ **Five-source Knowledge Base**: One KB containing Caldova's purchasing policy, both Search indexes, Fabric IQ, and Work IQ.
* ✓ **Executive supply-chain briefing**: A single query retrieves approved commitments, current actions, policy controls, supplier evidence, and medicinal-product dependencies.
* ✓ **Evidence-aware synthesis**: The answer distinguishes policy, contractual evidence, structured Fabric facts, and workplace discussions without inventing a Summit Dose-to-product link.